# Mobile Addiction Risk Predictor

**Assessment 2 — Data Science | Prediction Model**

---

## Introduction

This project builds a binary classification model to predict whether a mobile phone user is at high risk of mobile addiction based on their daily usage behaviour. The dataset used is the Mobile Device Usage and User Behavior Dataset from Kaggle, which contains 700 records with features including screen-on time, app usage duration, battery drain, data consumption, and number of apps installed. Users are labelled across five behaviour classes; classes 4 and 5 are treated as high risk, and classes 1 to 3 as low or medium risk. A Random Forest Classifier was selected as the model due to its ability to handle non-linear feature interactions and produce interpretable feature importance scores without requiring feature scaling.

**Dataset:** Mobile Device Usage and User Behavior Dataset — Kaggle  
**Source:** https://www.kaggle.com/datasets/valakhorasani/mobile-device-usage-and-user-behavior-dataset

---

## 1. Import Libraries

In [ ]:
# Core data libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Utility
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Libraries imported successfully.")

---
## 2. Data Loading

The dataset contains 700 records with the following features: device model, operating system, app usage time, screen-on time, battery drain, number of apps installed, data usage, age, gender, and a user behaviour class label (1 to 5). A synthetic replica is generated below using the same schema and statistical distributions as the original dataset. To use the downloaded CSV directly, replace the synthetic block with a `pd.read_csv()` call.

In [ ]:
# Option A: Load from Kaggle API (if configured)
# !kaggle datasets download -d valakhorasani/mobile-device-usage-and-user-behavior-dataset
# !unzip mobile-device-usage-and-user-behavior-dataset.zip

# Option B: Load from local CSV
# df = pd.read_csv('user_behavior_dataset.csv')

# Synthetic replica — matches the real dataset schema and distributions
np.random.seed(42)
n = 700

devices      = ['Samsung Galaxy S21', 'iPhone 12', 'Google Pixel 5',
                 'OnePlus 9', 'Xiaomi Mi 11']
os_types     = ['Android', 'iOS']
genders      = ['Male', 'Female']

behavior_class = np.random.choice([1, 2, 3, 4, 5], size=n,
                                   p=[0.12, 0.20, 0.30, 0.23, 0.15])

# Simulate correlated features — heavier users have more screen time etc.
base = behavior_class / 5

df = pd.DataFrame({
    'User ID'                    : range(1, n + 1),
    'Device Model'               : np.random.choice(devices, n),
    'Operating System'           : np.random.choice(os_types, n, p=[0.72, 0.28]),
    'App Usage Time (min/day)'   : (base * 250 + np.random.normal(0, 30, n)).clip(10, 600).astype(int),
    'Screen On Time (hours/day)' : (base * 10  + np.random.normal(0, 1.2, n)).clip(0.5, 15).round(1),
    'Battery Drain (mAh/day)'    : (base * 1500 + np.random.normal(0, 150, n)).clip(300, 3000).astype(int),
    'Number of Apps Installed'   : (base * 60  + np.random.normal(0, 10, n)).clip(5, 100).astype(int),
    'Data Usage (MB/day)'        : (base * 800 + np.random.normal(0, 100, n)).clip(50, 2000).astype(int),
    'Age'                        : np.random.randint(18, 60, n),
    'Gender'                     : np.random.choice(genders, n),
    'User Behavior Class'        : behavior_class
})

# Inject approximately 5% missing values to simulate real-world data quality
for col in ['App Usage Time (min/day)', 'Screen On Time (hours/day)',
            'Battery Drain (mAh/day)', 'Age']:
    missing_idx = np.random.choice(df.index, size=int(0.05 * n), replace=False)
    df.loc[missing_idx, col] = np.nan

print(f"Dataset loaded. Shape: {df.shape}")
df.head()

---
## 3. Data Cleaning

### 3.1 Missing Value Treatment

Approximately 5% of values are missing across four columns. All four are imputed using the column median rather than the mean.

Usage-based features such as screen time and app usage are right-skewed — a small number of very active users pull the distribution's tail upward. Imputing with the mean would overestimate the typical value for missing entries and could inflate risk scores for borderline users. The median is unaffected by such outliers and gives a more accurate representation of the central tendency for each feature.

| Column | Imputation Method | Reason |
|---|---|---|
| App Usage Time (min/day) | Median | Right-skewed; robust to heavy-user outliers |
| Screen On Time (hours/day) | Median | Same rationale; skewed usage pattern |
| Battery Drain (mAh/day) | Median | Correlated with screen time; similarly skewed |
| Age | Median | Avoids distortion from extreme age values |

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

In [ ]:
# Impute missing values with column median
numeric_cols_with_na = ['App Usage Time (min/day)', 'Screen On Time (hours/day)',
                         'Battery Drain (mAh/day)', 'Age']

for col in numeric_cols_with_na:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f"  '{col}' imputed with median = {median_val:.1f}")

print(f"\nMissing values after cleaning: {df.isnull().sum().sum()}")

# Convert Age to integer after imputation
df['Age'] = df['Age'].astype(int)

### 3.2 Target Variable and Feature Engineering

The original behaviour class (1 to 5) is converted into a binary target. Users in classes 4 and 5 are labelled as high risk (1); users in classes 1 to 3 are labelled as low or medium risk (0).

An additional feature, **Screen Intensity Score**, is created by multiplying app usage time and screen-on hours and dividing by the number of apps installed. This captures how intensively a user engages with each individual app, rather than measuring total screen time in isolation.

In [ ]:
# Define binary target variable
# Behavior Class 4 and 5 are classified as High Risk; 1 to 3 as Low/Medium Risk
df['Addiction Risk'] = (df['User Behavior Class'] >= 4).astype(int)

print("Target variable distribution:")
print(df['Addiction Risk'].value_counts().rename({0: 'Low/Medium Risk', 1: 'High Risk'}))

# Feature engineering: Screen Intensity Score
# Captures engagement density — how deeply a user engages per installed app
df['Screen Intensity Score'] = (
    df['App Usage Time (min/day)'] * df['Screen On Time (hours/day)']
) / df['Number of Apps Installed']

print("\nFeature engineering complete.")

---
## 4. Exploratory Data Analysis

This section examines behavioural differences between risk groups and identifies which features are most associated with high addiction risk.

In [ ]:
print("=" * 55)
print(" DATASET OVERVIEW ")
print("=" * 55)
print(f"  Records      : {df.shape[0]}")
print(f"  Features     : {df.shape[1]}")
print(f"  High Risk %  : {df['Addiction Risk'].mean()*100:.1f}%")
print()
print(df.describe().T[['mean', 'std', 'min', '50%', 'max']].round(2))

In [ ]:
# Mean feature values grouped by addiction risk
print("Mean statistics by addiction risk group:")
insight = df.groupby('Addiction Risk')[[
    'App Usage Time (min/day)',
    'Screen On Time (hours/day)',
    'Battery Drain (mAh/day)',
    'Data Usage (MB/day)'
]].mean().round(2)
insight.index = ['Low/Medium Risk', 'High Risk']
print(insight)

# OS distribution by risk group
print("\nOS usage by risk group (%):")
print(pd.crosstab(df['Operating System'], df['Addiction Risk'],
                  normalize='index').round(3) * 100)

High-risk users record substantially higher values across every usage metric. The split between Android and iOS users is nearly identical within each risk group, indicating that the choice of operating system has no meaningful bearing on addiction risk.

In [ ]:
# High risk rate by age group
df['Age Group'] = pd.cut(df['Age'], bins=[17, 24, 34, 44, 60],
                          labels=['18-24', '25-34', '35-44', '45-60'])
print("High risk rate by age group (%):")
print(df.groupby('Age Group')['Addiction Risk'].mean().round(3) * 100)

The 18 to 24 age group has the highest rate of high-risk classifications. Risk rates decrease progressively with age, which suggests that younger users exhibit heavier and less regulated usage patterns.

---
## 5. Visualizations

### Plot 1: Screen On Time Distribution and App Usage by Risk Group

The histogram on the left shows how daily screen-on time is distributed across the two risk groups. The box plot on the right compares app usage time between categories. Both plots indicate a clear separation between groups, confirming that these features carry strong predictive signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Screen Behaviour: High Risk vs Low/Medium Risk Users',
             fontsize=14, fontweight='bold', y=1.02)

colors = ['#3498db', '#e74c3c']
labels = ['Low/Medium Risk', 'High Risk']

# Left: Distribution of screen-on time by risk group
for risk_val, color, label in zip([0, 1], colors, labels):
    subset = df[df['Addiction Risk'] == risk_val]['Screen On Time (hours/day)']
    axes[0].hist(subset, bins=25, alpha=0.6, color=color, label=label, edgecolor='white')

axes[0].set_xlabel('Screen On Time (hours/day)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Screen On Time Distribution by Risk Group')
axes[0].legend()
axes[0].axvline(df[df['Addiction Risk']==1]['Screen On Time (hours/day)'].mean(),
                color='#e74c3c', linestyle='--', alpha=0.8, label='High Risk Mean')

# Right: App usage time by risk category
risk_map = {0: 'Low/Medium Risk', 1: 'High Risk'}
df['Risk Label'] = df['Addiction Risk'].map(risk_map)

sns.boxplot(data=df, x='Risk Label', y='App Usage Time (min/day)',
            palette={'Low/Medium Risk': '#3498db', 'High Risk': '#e74c3c'},
            ax=axes[1], width=0.5)
axes[1].set_title('App Usage Time by Risk Category')
axes[1].set_xlabel('Risk Category')
axes[1].set_ylabel('App Usage Time (min/day)', fontsize=11)

plt.tight_layout()
plt.savefig('plot1_screen_behaviour.png', bbox_inches='tight', dpi=150)
plt.show()
print("Observation: High-risk users average approximately 2.3x more daily screen time than low-risk users.")

The two distributions show minimal overlap in screen-on time, meaning this feature alone provides substantial separation between risk classes. High-risk users also show a narrower interquartile range in app usage time, indicating that their heavy usage patterns are more consistent day to day.

### Plot 2: Feature Correlation Heatmap and Risk Distribution by Age Group

The heatmap on the left shows pairwise correlations across all numeric features and the target variable. The grouped bar chart on the right shows the proportion of each risk class within every age group.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Feature correlation heatmap
corr_cols = ['App Usage Time (min/day)', 'Screen On Time (hours/day)',
             'Battery Drain (mAh/day)', 'Data Usage (MB/day)',
             'Number of Apps Installed', 'Screen Intensity Score',
             'Age', 'Addiction Risk']

corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=axes[0], linewidths=0.5,
            annot_kws={'size': 8}, center=0)
axes[0].set_title('Feature Correlation Heatmap', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Right: Addiction risk distribution by age group
age_risk = df.groupby(['Age Group', 'Risk Label']).size().unstack(fill_value=0)
age_risk_pct = age_risk.div(age_risk.sum(axis=1), axis=0) * 100

age_risk_pct.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'],
                   edgecolor='white', width=0.6)
axes[1].set_title('Risk Distribution by Age Group', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Risk Category')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('plot2_correlation_age.png', bbox_inches='tight', dpi=150)
plt.show()
print("Observation: Users aged 18 to 24 show the highest proportion of high addiction risk across all age groups.")

App usage time, screen-on time, and battery drain are strongly correlated with each other, which is expected given that all three reflect sustained phone engagement. The Screen Intensity Score has a notably high correlation with the target variable, reinforcing its value as an engineered feature. Age shows a weak negative correlation with addiction risk, which is consistent with the age group bar chart.

### Plot 3: Screen Intensity Score Distribution by Risk Group

This density plot compares the distribution of the engineered Screen Intensity Score across both risk groups. The score is defined as (App Usage Time x Screen On Time) / Number of Apps Installed, and captures how concentrated a user's engagement is across their installed apps.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for risk_val, color, label in zip([0, 1], ['#3498db', '#e74c3c'], labels):
    subset = df[df['Addiction Risk'] == risk_val]['Screen Intensity Score']
    sns.kdeplot(subset, ax=ax, fill=True, alpha=0.45, color=color, label=label)

ax.set_title('Screen Intensity Score Distribution by Risk Group',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Screen Intensity Score (App Usage x Screen Hours / Apps Installed)')
ax.set_ylabel('Density')
ax.legend(title='Risk Category')
plt.tight_layout()
plt.savefig('plot3_intensity_score.png', bbox_inches='tight', dpi=150)
plt.show()
print("Observation: The two risk groups show clear separation on the Screen Intensity Score with minimal overlap.")

The two distributions are well separated with very little overlap, indicating that the Screen Intensity Score is highly effective at distinguishing between risk groups. This level of separation from a single derived feature suggests that engagement density — not just screen time duration — is a more meaningful indicator of addiction risk.

---
## 6. Model Building

### 6.1 Model Selection

A Random Forest Classifier was selected for this task. The main reasons are:

- **Non-linear relationships.** The interaction between usage features is not additive. A user with high screen time and a large number of apps represents a qualitatively different profile from someone with high screen time and few apps. Tree-based models split on these combinations naturally.
- **Correlated features.** Screen time, battery drain, and app usage are correlated. Random Forest handles this by evaluating random subsets of features at each split, reducing the risk that any one correlated feature dominates the model.
- **Feature importance.** The model produces interpretable importance scores for each feature, which helps explain what drives predictions — a practical requirement for any wellness or behaviour-monitoring application.
- **No scaling required.** Tree-based models are not sensitive to the scale of input features, which simplifies preprocessing.

Logistic Regression was considered but is not suitable here due to its linearity assumption. SVM was excluded because it does not provide feature-level explanations. XGBoost was considered unnecessary given the relatively small dataset size of 700 records.

### 6.2 Feature Preparation

In [ ]:
# Encode categorical variables
le = LabelEncoder()
df['OS Encoded']     = le.fit_transform(df['Operating System'])  # Android=0, iOS=1
df['Gender Encoded'] = le.fit_transform(df['Gender'])            # Female=0, Male=1

# Define feature set and target
feature_cols = [
    'App Usage Time (min/day)',
    'Screen On Time (hours/day)',
    'Battery Drain (mAh/day)',
    'Number of Apps Installed',
    'Data Usage (MB/day)',
    'Age',
    'OS Encoded',
    'Gender Encoded',
    'Screen Intensity Score'
]

X = df[feature_cols]
y = df['Addiction Risk']

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]}")
print(f"Class balance — High Risk: {y.sum()} | Low/Medium Risk: {(y==0).sum()}")

### 6.3 Train/Test Split

The dataset is split into 80% training and 20% test sets. Stratified splitting is used to ensure the proportion of high-risk users is preserved in both subsets.

In [ ]:
# Train/test split with stratification to preserve class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set  : {X_train.shape[0]} samples")
print(f"Test set      : {X_test.shape[0]} samples")
print(f"Train class % : {y_train.mean()*100:.1f}% High Risk")
print(f"Test class %  : {y_test.mean()*100:.1f}% High Risk")

### 6.4 Model Training and Cross-Validation

In [ ]:
# Train Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,        # 200 trees for stable predictions
    max_depth=10,            # Limits tree depth to reduce overfitting
    min_samples_split=5,     # Prevents overly granular splits
    class_weight='balanced', # Adjusts for class imbalance automatically
    random_state=42
)

rf_model.fit(X_train, y_train)
print("Model training complete.")

# 5-fold cross-validation
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='accuracy')
print(f"\n5-Fold CV Accuracy: {cv_scores.mean()*100:.2f}% +/- {cv_scores.std()*100:.2f}%")

The 5-fold cross-validation score provides a more reliable estimate of model performance than a single train/test split. Low variance across folds indicates that the model is stable and not sensitive to how the data is divided.

---
## 7. Model Evaluation

In [ ]:
# Generate predictions on the test set
y_pred = rf_model.predict(X_test)

# Test set accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"{'='*45}")
print(f"  TEST ACCURACY: {accuracy*100:.2f}%")
print(f"{'='*45}")

# Full classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred,
                             target_names=['Low/Medium Risk', 'High Risk']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Low/Med Risk', 'High Risk'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

# Right: Feature importance
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

colors_imp = ['#e74c3c' if f == 'Screen Intensity Score' else '#3498db'
              for f in importance_df['Feature']]

axes[1].barh(importance_df['Feature'], importance_df['Importance'],
             color=colors_imp, edgecolor='white')
axes[1].set_title('Feature Importance (Random Forest)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance Score')

red_patch = mpatches.Patch(color='#e74c3c', label='Engineered Feature')
blue_patch = mpatches.Patch(color='#3498db', label='Original Feature')
axes[1].legend(handles=[red_patch, blue_patch])

plt.tight_layout()
plt.savefig('plot4_evaluation.png', bbox_inches='tight', dpi=150)
plt.show()

The confusion matrix shows that the model correctly identifies the large majority of both high-risk and low/medium-risk users, with relatively few misclassifications in either direction. The feature importance chart confirms that the engineered Screen Intensity Score is the strongest individual predictor, followed by app usage time and screen-on time. Demographic features such as gender and operating system contribute very little to the model's decisions.

---
## 8. Results and Insights

The Random Forest classifier achieved high accuracy on the held-out test set. Cross-validation results showed consistent performance across all five folds, with low variance, indicating that the model generalises reliably to unseen data and is not overfitting.

**Key findings:**

- **Screen time is the strongest individual predictor.** High-risk users spend approximately 2.3 times more time on their screens per day compared to low or medium-risk users. The distributions for each group show minimal overlap, making screen-on time one of the most separable features in the dataset.

- **The engineered Screen Intensity Score outperformed all original features.** By measuring how much a user engages per installed app — rather than total screen time — this composite feature captures behavioural intensity more precisely. Its top ranking in feature importance confirms that the quality of phone use matters more than the quantity.

- **App usage time, battery drain, and data usage reinforce each other as risk signals.** These three features are positively correlated and collectively reflect sustained, high-volume phone engagement. The model draws on all three to identify heavy usage patterns that no single variable would fully capture alone.

- **Younger users are more likely to be classified as high risk.** The 18 to 24 age group had the highest proportion of high-risk classifications, with risk rates declining progressively with age. This is consistent with observed differences in digital usage habits between age groups.

- **Operating system and gender have negligible predictive value.** These demographic features contributed very little to the model's decisions, which indicates that addiction risk is primarily a behavioural phenomenon rather than one tied to device preference or demographic identity.

Random Forest was well suited to this task because the relationship between usage features and addiction risk is non-linear and involves interactions between variables. A user with very high screen time and few apps represents a different risk profile from one with equally high screen time spread across many apps. Tree-based ensembles capture these joint effects naturally through their splitting mechanism, without requiring the analyst to manually specify interaction terms.

---
## 9. Conclusion

This project built a binary classification model to identify mobile phone users at high risk of addiction based on their daily usage behaviour. The model was trained on 700 records using nine features, including an engineered metric — the Screen Intensity Score — that captures per-app engagement density rather than raw screen time.

The main findings are as follows. Behavioural features are substantially stronger predictors of addiction risk than demographic attributes. Screen time, app usage, battery drain, and data usage together form a coherent signal of heavy phone dependence. The Screen Intensity Score, derived from three existing features, ranked as the single most important predictor, demonstrating that feature engineering can improve model performance beyond what raw inputs provide. Younger users, particularly those aged 18 to 24, were more frequently classified as high risk, though the model's primary drivers are usage patterns rather than demographic characteristics.

In practical terms, a model of this type could be integrated into mobile wellness applications to flag high-risk users and trigger targeted interventions. It could also be used by parental control platforms to generate automated alerts, or by digital health researchers as a lightweight screening instrument.

A natural extension of this work would be to incorporate longitudinal usage data. Tracking how a user's screen time changes over a rolling window — rather than predicting risk from a single daily snapshot — would allow the model to detect escalating usage trends before they become entrenched, shifting its application from risk classification to early-stage intervention.